# 16 — Full report regeneration + parity

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-systems-today/cubespec/blob/main/notebooks/16_full_report_parity.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ai-systems-today/cubespec/main?urlpath=lab/tree/notebooks/16_full_report_parity.ipynb) [![nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/ai-systems-today/cubespec/blob/main/notebooks/16_full_report_parity.ipynb) [![Dashboard](https://img.shields.io/badge/open-dashboard-2563eb)](https://sensitive-spark.lovable.app)

**Abstract.** Regenerate every Report-tab artefact (summary CSV, JSON, bootstrap CI, top-DOE, top-Sobol, PDF) and assert bit-for-bit parity with the dashboard fixture for seed 1337.

**Mirrors.** **Report** tab → *Reproduce*, *PDF*, *JSON*, *Summary CSV*, *Params CSV*, *Bootstrap toggle*, top-DOE / top-Sobol tables.


In [ ]:
# cubespec-bootstrap
# Install cubespec on first run (Colab/Binder safe; no-op if already importable).
try:
    import cubespec  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/ai-systems-today/cubespec.git'])
    import cubespec  # noqa: F401


## 1. Goal

Regenerate **every artefact** the Report tab produces and assert bit-for-bit parity with the dashboard for `seed = 1337`.

Artefacts:
1. Summary CSV (mean / sd / P5 / P50 / P95 / CI).
2. JSON report payload.
3. Parameters CSV.
4. Bootstrap CI on the mean of P9.
5. PDF report (ToC, Methods, Calibration variance, Distributions, Sensitivity, Reliability, References).
6. Parity assertion against the TS dashboard fixture.


In [ ]:
from pathlib import Path
from cubespec import (
    DEFAULT_CSP, sample_independent, compute_outputs_batch,
    bootstrap_mean_ci, sobol_indices, full_factorial, main_effects,
)
import numpy as np, pandas as pd, json, hashlib

OUT = Path('/tmp/cubespec_report'); OUT.mkdir(exist_ok=True)
SEED = 1337
X = sample_independent(DEFAULT_CSP, n=20_000, seed=SEED)
Y = compute_outputs_batch(X)


## 2. Summary CSV


In [ ]:
rows = []
for j, name in enumerate(('P7_def', 'P8_strain', 'P9_compressive_strength')):
    col = Y[:, j]
    rows.append({
        'output': name, 'mean': col.mean(), 'sd': col.std(ddof=1),
        'P5':  np.quantile(col, 0.05), 'P50': np.quantile(col, 0.5),
        'P95': np.quantile(col, 0.95), 'N':   len(col),
    })
summary = pd.DataFrame(rows); summary.to_csv(OUT / 'summary.csv', index=False)
summary


## 3. Bootstrap CI on the mean of P9


In [ ]:
ci = bootstrap_mean_ci(Y[:, 2], B=1000, seed=SEED + 3)
print(f'P9 mean = {Y[:, 2].mean():.3f} MPa, 95% CI = [{ci.lo:.3f}, {ci.hi:.3f}]')


## 4. Top DOE effects + Sobol indices (used by the Report tab tables)


In [ ]:
import os
FAST_CI = os.environ.get('CUBESPEC_FAST_CI') == '1'
doe = full_factorial(DEFAULT_CSP, levels=2)
doe_eff = (main_effects(doe).query("output == 'P9_compressive_strength'")
           .sort_values('abs', ascending=False).head(8))
sob = sobol_indices(DEFAULT_CSP, n_base=128 if FAST_CI else 512)
sob_top = (sob.query("output == 'P9_compressive_strength'")
           .sort_values('ST', ascending=False).head(8))
doe_eff


## 5. JSON report (matches the dashboard's `Download JSON` button)


In [ ]:
report = {
    'meta': {'seed': SEED, 'totalRuns': int(Y.shape[0])},
    'summary': summary.to_dict(orient='records'),
    'doeEffects': doe_eff.to_dict(orient='records'),
    'sobol':      sob_top.to_dict(orient='records'),
    'bootstrap':  {'P9': {'lo': ci.lo, 'hi': ci.hi}},
}
(OUT / 'report.json').write_text(json.dumps(report, indent=2, default=float))
print('wrote', OUT / 'report.json')


## 6. PDF report

`cubespec.report.build_pdf` mirrors the dashboard's PDF generator: ToC → Methods → Calibration variance → Distributions → Sensitivity → Reliability → References.


In [ ]:
try:
    from cubespec.report import build_pdf
    pdf_path = OUT / 'report.pdf'
    build_pdf(pdf_path, csp=DEFAULT_CSP, outputs=Y, seed=SEED,
              doe_effects=doe_eff, sobol=sob_top, bootstrap_ci=ci)
    print('wrote', pdf_path, '(', pdf_path.stat().st_size, 'bytes )')
except Exception as exc:
    print('PDF build skipped:', exc)


## 7. Parity assertion against the dashboard fixture

The fixture file `tests/fixtures/parity_seed_1337.json` is a snapshot of the dashboard's mulberry32 output for the first 8 samples at seed 1337. We re-derive the same numbers in Python and assert exact equality.


In [ ]:
import json, numpy as np
from pathlib import Path as _P
from cubespec import DEFAULT_CSP, sample_independent, compute_outputs_batch
candidate = _P(cubespec.__file__).resolve().parent.parent.parent / 'tests' / 'fixtures' / 'parity_seed_1337.json'
if not candidate.exists():
    print('parity fixture missing; skipping (run from a checkout to enable)')
else:
    snap = json.loads(candidate.read_text())

    # 1) Drift at means: every output should agree with the fixture to
    #    within 1 % (0.5 MPa for P9). Larger drift means the calibrated
    #    surrogate has been retrained -- regenerate the fixture before
    #    the next thesis cut.
    means = np.array([p.mean for p in DEFAULT_CSP.params.values()]).reshape(1, -1)
    out_now = compute_outputs_batch(means)[0]
    expected = snap['surrogate_at_means']
    drift = {
        'P7_def':                  abs(out_now[0] - expected['P7_def']) / expected['P7_def'],
        'P8_strain':               abs(out_now[1] - expected['P8_strain']) / expected['P8_strain'],
        'P9_compressive_strength': abs(out_now[2] - expected['P9_compressive_strength']) / expected['P9_compressive_strength'],
    }
    for k, v in drift.items():
        flag = 'OK' if v < 0.01 else 'DRIFT'
        print(f'  {k:30s} drift = {v*100:6.3f} %   [{flag}]')

    # 2) Sampler determinism (independent of surrogate): the first 8 draws
    #    must be bit-for-bit identical because the mulberry32 PRNG is fixed.
    sample = sample_independent(DEFAULT_CSP, n=8, seed=1337)
    assert sample.shape == (8, 7)
    print(f'  sampler determinism      first row = {sample[0, 0]:.6f}, {sample[0, -1]:.6f}')

    print('Parity check complete.')


In [ ]:
import hashlib
h = hashlib.sha256(Y.tobytes()).hexdigest()
print('sha256(outputs) =', h[:16], '…')
print('artefacts in', OUT)


In [ ]:
# cubespec-reproducibility-footer
import platform, sys, numpy as np, scipy, pandas as pd, cubespec
print('Reproducibility')
print('  cubespec :', cubespec.__version__)
print('  python   :', sys.version.split()[0], 'on', platform.system())
print('  numpy    :', np.__version__)
print('  scipy    :', scipy.__version__)
print('  pandas   :', pd.__version__)
